In [0]:
from pyspark.sql import functions as F


In [0]:
silver_df = spark.table(
    "medallion_catalog.silver.silver_orders"
)

In [0]:
gold_df = (
    silver_df
    .groupBy("city")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("amount").alias("total_sales"),
        F.avg("amount").alias("average_order_value")
    )
)

In [0]:
gold_df.show()

In [0]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "medallion_catalog.gold.sales_by_city"
    )

In [0]:
%sql

SELECT *
FROM medallion_catalog.gold.sales_by_city
ORDER BY total_sales DESC;

In [0]:
gold_status_df = (
    silver_df
    .groupBy("status")
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("amount").alias("total_sales"),
        F.avg("amount").alias("average_order_value")
    )
    .withColumn(
        "average_order_value",
        F.round("average_order_value", 2)
    )
)

In [0]:
gold_status_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "medallion_catalog.gold.sales_by_status"
    )

In [0]:
%sql

SELECT *
FROM medallion_catalog.gold.sales_by_status
ORDER BY total_sales DESC;